# MongoDB Candle Data Sampler

Pulls sample OHLCV candle data from MongoDB, inspects schema/quality, and writes
the full diagnostic report to `candle_sample_output.txt` for offline review.

**Edit the CONFIG cell below**, then **Run All**.

In [1]:
import os

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — edit these or rely on environment variables
# ─────────────────────────────────────────────────────────────────────────────
TRUENAS_IP      = os.environ.get("TRUENAS_LAN_IP", "192.168.1.54")
TRUENAS_DB_PASS = os.environ.get("MONGO_ROOT_PASSWORD", "mypass")
MONGO_URI = os.environ.get(
    "MONGO_URI",
    f"mongodb://admin:{TRUENAS_DB_PASS}@{TRUENAS_IP}:27017/quants_lab?authSource=admin",
)
MONGO_DATABASE = os.environ.get("MONGO_DATABASE", "quants_lab")

CONNECTOR    = "nonkyc"
TRADING_PAIR = "BTC-USDT"    # MongoDB format (hyphen, not slash)
INTERVAL     = "5m"

N_SAMPLES    = 10             # rows to show per section
OUTPUT_FILE  = "candle_sample_output.txt"

print(f"Target: {CONNECTOR} {TRADING_PAIR} {INTERVAL}")
print(f"Output: {OUTPUT_FILE}")

Target: nonkyc BTC-USDT 5m
Output: candle_sample_output.txt


In [2]:
import sys, json
from datetime import datetime, timezone
from io import StringIO

import pandas as pd
from pymongo import MongoClient

# ── helpers ──
def ts_to_str(ts_val):
    try:
        if isinstance(ts_val, (int, float)):
            if ts_val > 1e12:
                return datetime.fromtimestamp(ts_val / 1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M")
            return datetime.fromtimestamp(ts_val, tz=timezone.utc).strftime("%Y-%m-%d %H:%M")
        return str(ts_val)
    except Exception:
        return str(ts_val)

def serialize(doc):
    out = {}
    for k, v in doc.items():
        if isinstance(v, datetime):
            out[k] = v.isoformat()
        elif isinstance(v, bytes):
            out[k] = v.hex()
        else:
            out[k] = v
    return out

# We'll capture everything into `buf` so it prints to the notebook AND saves to file.
buf = StringIO()

def tee(*args, **kwargs):
    """Print to both stdout and the capture buffer."""
    import builtins
    text = " ".join(str(a) for a in args)
    end = kwargs.get("end", "\n")
    builtins.print(text, end=end)
    buf.write(text + end)

# ── connect ──
safe_uri = MONGO_URI.split("@")[-1] if "@" in MONGO_URI else MONGO_URI
tee("=" * 70)
tee("MongoDB Candle Data Sampler")
tee("=" * 70)
tee(f"  URI:      ...@{safe_uri}")
tee(f"  Database: {MONGO_DATABASE}")
tee()

client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
client.admin.command("ping")
tee("  ✓ Connected successfully")
tee()

db = client[MONGO_DATABASE]

MongoDB Candle Data Sampler
  URI:      ...@192.168.1.54:27017/quants_lab?authSource=admin&retryWrites=true&w=majority
  Database: quants_lab

  ✓ Connected successfully



In [3]:
tee("─" * 70)
tee("SECTION 1: COLLECTIONS IN DATABASE")
tee("─" * 70)
for coll_name in sorted(db.list_collection_names()):
    count = db[coll_name].estimated_document_count()
    tee(f"  {coll_name:30s}  ~{count:,} docs")
tee()

──────────────────────────────────────────────────────────────────────
SECTION 1: COLLECTIONS IN DATABASE
──────────────────────────────────────────────────────────────────────
  candles                         ~4,504,079 docs
  market_trades                   ~0 docs



In [4]:
tee("─" * 70)
tee("SECTION 2: AVAILABLE CONNECTOR / PAIR / INTERVAL COMBOS")
tee("─" * 70)

pipeline = [
    {"$group": {
        "_id": {
            "connector": "$connector",
            "trading_pair": "$trading_pair",
            "interval": "$interval",
        },
        "count": {"$sum": 1},
        "min_ts": {"$min": "$timestamp"},
        "max_ts": {"$max": "$timestamp"},
    }},
    {"$sort": {"_id.connector": 1, "_id.trading_pair": 1, "_id.interval": 1}},
]
combos = list(db.candles.aggregate(pipeline))
for c in combos:
    cid = c["_id"]
    min_dt = ts_to_str(c["min_ts"])
    max_dt = ts_to_str(c["max_ts"])
    tee(f"  {cid['connector']:12s} {cid['trading_pair']:15s} {cid['interval']:5s} "
        f" {c['count']:>8,} candles   {min_dt} → {max_dt}")
tee()

──────────────────────────────────────────────────────────────────────
SECTION 2: AVAILABLE CONNECTOR / PAIR / INTERVAL COMBOS
──────────────────────────────────────────────────────────────────────
  coinbase     BTC-USD         15m      17,322 candles   2025-09-09 07:45 → 2026-03-08 18:00
  coinbase     BTC-USD         1d          180 candles   2025-09-09 00:00 → 2026-03-07 00:00
  coinbase     BTC-USD         1h        4,331 candles   2025-09-09 07:00 → 2026-03-08 17:00
  coinbase     BTC-USD         1m      259,848 candles   2025-09-09 07:31 → 2026-03-08 18:18
  coinbase     BTC-USD         5m       51,966 candles   2025-09-09 07:45 → 2026-03-08 18:10
  coinbase     ETH-USD         15m      17,322 candles   2025-09-09 07:45 → 2026-03-08 18:00
  coinbase     ETH-USD         1d          180 candles   2025-09-09 00:00 → 2026-03-07 00:00
  coinbase     ETH-USD         1h        4,331 candles   2025-09-09 07:00 → 2026-03-08 17:00
  coinbase     ETH-USD         1m      259,834 candles   2

In [5]:
query = {
    "connector": CONNECTOR,
    "trading_pair": TRADING_PAIR,
    "interval": INTERVAL,
}

tee("─" * 70)
tee(f"SECTION 3: RAW MONGODB DOCUMENTS — FIRST {N_SAMPLES}")
tee("─" * 70)
first_docs = list(db.candles.find(query, {"_id": 0}).sort("timestamp", 1).limit(N_SAMPLES))
if not first_docs:
    tee(f"  ⚠ No documents found for {CONNECTOR} {TRADING_PAIR} {INTERVAL}")
    tee("  Check the combos listed in Section 2 and update CONFIG cell.")
else:
    for i, doc in enumerate(first_docs):
        tee(f"  [{i}] {json.dumps(serialize(doc), default=str)}")

tee()
tee("─" * 70)
tee(f"SECTION 4: RAW MONGODB DOCUMENTS — LAST {N_SAMPLES}")
tee("─" * 70)
last_docs = list(db.candles.find(query, {"_id": 0}).sort("timestamp", -1).limit(N_SAMPLES))
last_docs.reverse()
for i, doc in enumerate(last_docs):
    tee(f"  [{i}] {json.dumps(serialize(doc), default=str)}")
tee()

──────────────────────────────────────────────────────────────────────
SECTION 3: RAW MONGODB DOCUMENTS — FIRST 10
──────────────────────────────────────────────────────────────────────
  [0] {"timestamp": 1756833000, "connector": "nonkyc", "interval": "5m", "trading_pair": "BTC-USDT", "close": 110657.72, "high": 110819.73, "low": 110622.08, "open": 110791.88, "volume": 0.463, "schema_version": 2, "open_ts": 1756833000, "base_asset": "BTC", "quote_asset": "USDT", "base_volume": 0.463, "close_ts": 1756833300, "updated_at": 1772953582, "ingested_at": 1772953582, "quote_volume": 51254.02746333334, "quote_volume_is_estimated": true, "qc_ok": true, "qc_flags": [], "is_closed": true}
  [1] {"timestamp": 1756833300, "connector": "nonkyc", "interval": "5m", "trading_pair": "BTC-USDT", "close": 110693.33, "high": 111118.37, "low": 110650.59, "open": 110669.88, "volume": 0.215, "schema_version": 2, "open_ts": 1756833300, "base_asset": "BTC", "quote_asset": "USDT", "base_volume": 0.215, "close_ts

In [6]:
tee("─" * 70)
tee("SECTION 5: SCHEMA ANALYSIS")
tee("─" * 70)
if first_docs:
    sample_doc = first_docs[0]
    tee("  Fields and types in first document:")
    for key, val in sample_doc.items():
        tee(f"    {key:20s}  type={type(val).__name__:10s}  value={val!r}")

    ts_val = sample_doc.get("timestamp")
    tee(f"\n  Timestamp field raw value: {ts_val!r}")
    if isinstance(ts_val, (int, float)):
        if ts_val > 1e12:
            tee(f"    → MILLISECONDS (÷1000 → {datetime.fromtimestamp(ts_val/1000, tz=timezone.utc)})")
        elif ts_val > 1e9:
            tee(f"    → SECONDS (→ {datetime.fromtimestamp(ts_val, tz=timezone.utc)})")
        else:
            tee(f"    → Unexpected magnitude: {ts_val}")
    elif isinstance(ts_val, datetime):
        tee(f"    → Native datetime object: {ts_val}")
    else:
        tee(f"    → Unexpected type: {type(ts_val)}")
tee()

──────────────────────────────────────────────────────────────────────
SECTION 5: SCHEMA ANALYSIS
──────────────────────────────────────────────────────────────────────
  Fields and types in first document:
    timestamp             type=int         value=1756833000
    connector             type=str         value='nonkyc'
    interval              type=str         value='5m'
    trading_pair          type=str         value='BTC-USDT'
    close                 type=float       value=110657.72
    high                  type=float       value=110819.73
    low                   type=float       value=110622.08
    open                  type=float       value=110791.88
    volume                type=float       value=0.463
    schema_version        type=int         value=2
    open_ts               type=int         value=1756833000
    base_asset            type=str         value='BTC'
    quote_asset           type=str         value='USDT'
    base_volume           type=float       value

In [7]:
tee("─" * 70)
tee("SECTION 6: SORT ORDER & TIMESTAMP GAP CHECK")
tee("─" * 70)
if first_docs:
    all_ts = [d["timestamp"] for d in first_docs]
    sorted_asc = all(all_ts[i] <= all_ts[i + 1] for i in range(len(all_ts) - 1))
    tee(f"  First {len(all_ts)} timestamps ascending? {sorted_asc}")
    if len(all_ts) >= 2:
        gaps = [all_ts[i + 1] - all_ts[i] for i in range(len(all_ts) - 1)]
        tee(f"  Gaps between consecutive timestamps: {gaps}")
        if all(isinstance(g, (int, float)) for g in gaps):
            tee(f"  Min gap: {min(gaps)},  Max gap: {max(gaps)},  Median: {sorted(gaps)[len(gaps)//2]}")
tee()

──────────────────────────────────────────────────────────────────────
SECTION 6: SORT ORDER & TIMESTAMP GAP CHECK
──────────────────────────────────────────────────────────────────────
  First 10 timestamps ascending? True
  Gaps between consecutive timestamps: [300, 300, 300, 300, 300, 300, 300, 300, 300]
  Min gap: 300,  Max gap: 300,  Median: 300



In [8]:
tee("─" * 70)
tee("SECTION 7: DATA QUALITY — DUPLICATES & NULLS")
tee("─" * 70)

total_count = db.candles.count_documents(query)
tee(f"  Total documents matching query: {total_count:,}")

# Duplicates
dup_pipeline = [
    {"$match": query},
    {"$group": {"_id": "$timestamp", "count": {"$sum": 1}}},
    {"$match": {"count": {"$gt": 1}}},
    {"$count": "n_duplicates"},
]
dup_result = list(db.candles.aggregate(dup_pipeline))
n_dups = dup_result[0]["n_duplicates"] if dup_result else 0
tee(f"  Duplicate timestamps: {n_dups}")

# Nulls / missing fields
for col in ["open", "high", "low", "close", "volume"]:
    null_q = {**query, col: {"$in": [None]}}
    n_null = db.candles.count_documents(null_q)
    missing_q = {**query, col: {"$exists": False}}
    n_missing = db.candles.count_documents(missing_q)
    status = "✓ OK" if (n_null == 0 and n_missing == 0) else f"⚠ {n_null} null, {n_missing} missing"
    tee(f"  {col:8s}: {status}")
tee()

──────────────────────────────────────────────────────────────────────
SECTION 7: DATA QUALITY — DUPLICATES & NULLS
──────────────────────────────────────────────────────────────────────
  Total documents matching query: 53,880
  Duplicate timestamps: 0
  open    : ✓ OK
  high    : ✓ OK
  low     : ✓ OK
  close   : ✓ OK
  volume  : ✓ OK



In [9]:
tee("─" * 70)
tee("SECTION 8: PANDAS DATAFRAME PREVIEW (first 20 rows)")
tee("─" * 70)
cursor = db.candles.find(query, {"_id": 0}).sort("timestamp", 1).limit(20)
rows = list(cursor)
df = pd.DataFrame(rows)
if "timestamp" in df.columns:
    ts_col = df["timestamp"]
    if ts_col.dtype in ("int64", "float64") and len(ts_col) > 0 and ts_col.iloc[0] > 1e9:
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

for col in ("open", "high", "low", "close", "volume"):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

drop_cols = [c for c in ("connector", "trading_pair", "interval") if c in df.columns]
df_display = df.drop(columns=drop_cols)

tee(f"  Shape: {df_display.shape}")
tee(f"  Dtypes:")
tee(df_display.dtypes.to_string())
tee()
tee(df_display.to_string(index=True))
tee()
tee("  Describe:")
tee(df_display.describe().to_string())
tee()

──────────────────────────────────────────────────────────────────────
SECTION 8: PANDAS DATAFRAME PREVIEW (first 20 rows)
──────────────────────────────────────────────────────────────────────
  Shape: (20, 19)
  Dtypes:
timestamp                    datetime64[s, UTC]
close                                   float64
high                                    float64
low                                     float64
open                                    float64
volume                                  float64
schema_version                            int64
open_ts                                   int64
base_asset                                  str
quote_asset                                 str
base_volume                             float64
close_ts                                  int64
updated_at                                int64
ingested_at                               int64
quote_volume                            float64
quote_volume_is_estimated                  bool
qc_ok     

In [10]:
tee("─" * 70)
tee(f"SECTION 9: MID-SECTION SAMPLE ({N_SAMPLES} docs from ~middle of dataset)")
tee("─" * 70)
skip_count = max(0, total_count // 2 - N_SAMPLES // 2)
mid_docs = list(
    db.candles.find(query, {"_id": 0})
    .sort("timestamp", 1)
    .skip(skip_count)
    .limit(N_SAMPLES)
)
for i, doc in enumerate(mid_docs):
    tee(f"  [{i}] {json.dumps(serialize(doc), default=str)}")
tee()

──────────────────────────────────────────────────────────────────────
SECTION 9: MID-SECTION SAMPLE (10 docs from ~middle of dataset)
──────────────────────────────────────────────────────────────────────
  [0] {"connector": "nonkyc", "interval": "5m", "trading_pair": "BTC-USDT", "timestamp": 1764913500, "close": 91945.58, "high": 92025.18, "low": 91930.84, "open": 92011.14, "volume": 0.212, "schema_version": 2, "open_ts": 1764913500, "base_asset": "BTC", "quote_asset": "USDT", "base_volume": 0.212, "close_ts": 1764913800, "updated_at": 1772953582, "ingested_at": 1772953582, "quote_volume": 19497.0464, "quote_volume_is_estimated": true, "qc_ok": true, "qc_flags": [], "is_closed": true}
  [1] {"connector": "nonkyc", "interval": "5m", "trading_pair": "BTC-USDT", "timestamp": 1764913800, "close": 91888.67, "high": 91959.99, "low": 91888.67, "open": 91945.58, "volume": 0.09, "schema_version": 2, "open_ts": 1764913800, "base_asset": "BTC", "quote_asset": "USDT", "base_volume": 0.09, "close

In [11]:
client.close()

tee("=" * 70)
tee("Done.")
tee("=" * 70)

# ── Write captured output to file ──
with open(OUTPUT_FILE, "w") as f:
    f.write(buf.getvalue())

print(f"\n>>> Full report saved to: {OUTPUT_FILE}")
print(f"    ({len(buf.getvalue()):,} characters, {buf.getvalue().count(chr(10)):,} lines)")

Done.

>>> Full report saved to: candle_sample_output.txt
    (47,371 characters, 391 lines)
